# 04 - Figures Paper 4 XAI (PNG untuk .tex)

Menghasilkan 4 gambar PNG dari artefak hasil (notebook 01-03) untuk `paper4-xai.tex`:
`fig1_pipeline.png`, `fig2_f1_vs_mcc.png`, `fig3_fewshot.png`, `fig4_shap.png`.

Dijalankan di AWS SageMaker. Artefak JSON di-sync dari S3; PNG ditulis ke `../figure-p4/`
dan di-upload ke S3. Semua angka dari eksperimen nyata.

In [ ]:
import sys
!{sys.executable} -m pip install -q matplotlib numpy
print('deps OK')

In [ ]:
import os, json, subprocess
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

ART='./artifacts'
S3_ARTIFACTS_OUT='s3://ssh-detection-features-232032302717/paper4-xai/artifacts/'
FIGDIR='../figure-p4'
os.makedirs(FIGDIR, exist_ok=True)
os.makedirs(ART, exist_ok=True)
# selalu tarik artefak terbaru
subprocess.run(['aws','s3','sync',S3_ARTIFACTS_OUT,ART,'--only-show-errors'], check=True)
diag=json.load(open(f'{ART}/diagnosis_results.json'))
shap=json.load(open(f'{ART}/shap_results.json'))
def M(kind,key): return diag['summary'][kind][key]['mean']
print('artefak dimuat. models:', diag['models'])

In [ ]:
# === Fig 1: diagram pipeline bebas-kebocoran ===
fig, ax = plt.subplots(figsize=(10,2.6)); ax.axis('off')
steps=['Normalisasi\nkolom & label','Pembersihan\n+ imputasi\n(median train)',
       'Irisan nama\nfitur (tanpa SFM)','Split 80/20\nstratified',
       'Fit train-only:\nseleksi fitur,\nscaler, SMOTE','Latih model\n(XGB / RF)']
n=len(steps); x=np.linspace(0.04,0.78,n)
for i,(xi,s) in enumerate(zip(x,steps)):
    ax.add_patch(plt.Rectangle((xi,0.45),0.13,0.32,fc='#dfe9f5',ec='#2a4d7a',lw=1.3))
    ax.text(xi+0.065,0.61,s,ha='center',va='center',fontsize=7.5)
    if i<n-1: ax.annotate('',xy=(x[i+1],0.61),xytext=(xi+0.13,0.61),arrowprops=dict(arrowstyle='->',lw=1.1))
# transfer arrow
ax.add_patch(plt.Rectangle((0.84,0.45),0.13,0.32,fc='#f5e6df',ec='#7a3b2a',lw=1.3))
ax.text(0.905,0.61,'Transfer ke\ntarget 2018\n(tanpa re-fit)',ha='center',va='center',fontsize=7.5)
ax.annotate('',xy=(0.84,0.61),xytext=(x[-1]+0.13,0.61),arrowprops=dict(arrowstyle='->',lw=1.1))
ax.text(0.5,0.20,'Semua operasi fitting HANYA pada data latih sumber (CIC-IDS2017)',
        ha='center',fontsize=8,style='italic',color='#444')
ax.set_xlim(0,1.02); ax.set_ylim(0,1)
plt.tight_layout(); plt.savefig(f'{FIGDIR}/fig1_pipeline.png',dpi=200,bbox_inches='tight'); plt.close()
print('fig1 saved')

In [ ]:
# === Fig 2: weighted-F1 vs MCC (within vs cross) ===
models=['xgb','rf']; labels=['XGBoost','Random Forest']
f1_w=[M(k,'within_ref_f1') for k in models]; mcc_w=[M(k,'within_ref_mcc') for k in models]
f1_c=[M(k,'ref2tgt_f1') for k in models]; mcc_c=[M(k,'ref2tgt_mcc') for k in models]
x=np.arange(len(models)); w=0.2
fig,ax=plt.subplots(figsize=(7,4))
ax.bar(x-1.5*w,f1_w,w,label='F1 within',color='#9ecae1')
ax.bar(x-0.5*w,mcc_w,w,label='MCC within',color='#3182bd')
ax.bar(x+0.5*w,f1_c,w,label='F1 cross',color='#fdae6b')
ax.bar(x+1.5*w,mcc_c,w,label='MCC cross',color='#e6550d')
ax.set_xticks(x); ax.set_xticklabels(labels); ax.set_ylabel('Score'); ax.set_ylim(0,1.05)
ax.set_title('Weighted-F1 tetap tinggi, MCC runtuh lintas-dataset (CIC2017->CIC2018)')
ax.legend(fontsize=8,ncol=2); ax.grid(axis='y',ls=':',alpha=0.5)
plt.tight_layout(); plt.savefig(f'{FIGDIR}/fig2_f1_vs_mcc.png',dpi=200,bbox_inches='tight'); plt.close()
print('fig2 saved')

In [ ]:
# === Fig 3: kurva few-shot (MCC vs % label target) ===
fr=diag['fracs']; xf=[f*100 for f in fr]
fig,ax=plt.subplots(figsize=(7,4))
series=[('xgb','fs_ref2tgt','XGB 2017->2018','o-','#e6550d'),
        ('rf','fs_ref2tgt','RF 2017->2018','s-','#fd8d3c'),
        ('xgb','fs_tgt2ref','XGB 2018->2017','o--','#3182bd'),
        ('rf','fs_tgt2ref','RF 2018->2017','s--','#9ecae1')]
for kind,base,lab,st,col in series:
    ys=[diag['summary'][kind][f'{base}_{f}']['mean'] for f in fr]
    ax.plot(xf,ys,st,label=lab,color=col,lw=1.6,ms=5)
ax.set_xlabel('Persentase label target (%)'); ax.set_ylabel('MCC'); ax.set_ylim(-0.05,1.05)
ax.set_title('Pemulihan few-shot: 1% label target memulihkan deteksi')
ax.legend(fontsize=8); ax.grid(ls=':',alpha=0.5)
plt.tight_layout(); plt.savefig(f'{FIGDIR}/fig3_fewshot.png',dpi=200,bbox_inches='tight'); plt.close()
print('fig3 saved')

In [ ]:
# === Fig 4: SHAP top-8 fitur (gagal vs pulih) ===
fig,axes=plt.subplots(1,2,figsize=(12,4.5))
for ax,kind,title in zip(axes,['xgb','rf'],['XGBoost','Random Forest']):
    rf_=shap[kind]['rank_fail']; rh_=shap[kind]['rank_heal']
    feats=list(rf_)[:8][::-1]
    vf=[rf_[f] for f in feats]; vh=[rh_.get(f,0) for f in feats]
    y=np.arange(len(feats)); h=0.4
    ax.barh(y+h/2,vf,h,label='Gagal (transfer)',color='#de2d26')
    ax.barh(y-h/2,vh,h,label='Pulih (few-shot 1%)',color='#31a354')
    ax.set_yticks(y); ax.set_yticklabels(feats,fontsize=7)
    ax.set_xlabel('mean |SHAP|'); ax.set_title(f'{title} (TreeSHAP)'); ax.legend(fontsize=7)
    ax.grid(axis='x',ls=':',alpha=0.5)
plt.suptitle('SHAP sadar-kondisi: stabilitas fitur tidak menjamin transferabilitas',fontsize=10)
plt.tight_layout(); plt.savefig(f'{FIGDIR}/fig4_shap.png',dpi=200,bbox_inches='tight'); plt.close()
print('fig4 saved')

In [ ]:
# === Upload PNG ke S3 (opsional, untuk arsip) ===
subprocess.run(['aws','s3','sync',FIGDIR,'s3://ssh-detection-features-232032302717/paper4-xai/figure-p4/',
                '--only-show-errors','--exclude','*','--include','*.png'], check=True)
print('PNG ter-upload. Juga ter-commit via git dari ../figure-p4/ untuk Overleaf.')
import glob; print(glob.glob(f'{FIGDIR}/*.png'))